#### 03. Train-validation-test tables

This notebook joins the labeled observations with the Tier 0 features and writes train, validation, and test tables.

**Inputs** (from notebooks 01 and 02)
- `task2/data/labels/`: one row per (station, 5-minute slot) with the 30-minute future rainfall label
- `task2/data/station_static.csv`: station position and O/E wetness ratios
- `task2/code/features.py`: `add_time_features`, shared with training and prediction

Here, we provide the next cell with one optional configuration, `SUBSAMPLE = True` keeps every wet row and a share `FRAC` of the dry rows (y_mm <= 0); `SUBSAMPLE = False` keeps every training row. The two training tables go to different folders, so both can exist side by side and be compared.

**Outputs**: `task2/data/assembled/`
* train_subsampled (2017-2022) `train_frac0.1/`, for this training sample, we subsample specifically rows
with label 0, meaning no rainfall observed in the next 30 minutes, to keep 10% of its original number of observations. 
* train (2017-2022) `train_full/`, a full training sample without subsampling, available for machines with higher computational capacity.
* validation set (2023) `val/`
* test set (2024) `test/`

In [29]:
SUBSAMPLE = True   # True:  keep all wet rows and a share FRAC of the dry rows 
FRAC = 0.1         
SEED = 42          

assert 0 < FRAC <= 1
TRAIN_NAME = f"train_frac{FRAC:g}" if SUBSAMPLE else "train_full"
print("training table:", TRAIN_NAME)

training table: train_frac0.1


In [30]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parents[1]
MATRIX = ROOT / "task1_revision" / "data" / "matrix_5min"
OUT = ROOT / "task2" / "data"
ASSEMBLED = OUT / "assembled"
assert (OUT / "labels").exists(), OUT

sys.path.append(str(ROOT / "task2" / "code"))
from features import add_time_features

from pyspark.sql import SparkSession, functions as F, Window

spark = (SparkSession.builder
         .master("local[8]")
         .appName("task2-assemble")
         .config("spark.driver.memory", "10g")
         .config("spark.sql.session.timeZone", "Asia/Singapore")
         .config("spark.sql.shuffle.partitions", "96")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

In [31]:
# First read the station specific features from the small csv
static_pdf = pd.read_csv(OUT / "station_static.csv")
STATION_FEATURES = ["t0_x_km", "t0_y_km", "t0_wet_ratio", "t0_heavy_ratio"]
static = spark.createDataFrame(static_pdf[["station_id"] + STATION_FEATURES])

assert len(static_pdf) == 87 and static_pdf[STATION_FEATURES].notna().all().all()

/Users/lynnchen/anaconda3/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:687: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [UNSUPPORTED_PACKAGE_VERSION] PyArrow >= 18.0.0 must be installed; however, your version is 16.1.0.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


In [32]:
# Read our labeled data and split into train/valid/test sets
labels = spark.read.parquet(str(OUT / "labels"))

train = labels.filter(F.col("year") <= 2022)
val   = labels.filter(F.col("year") == 2023)
test  = labels.filter(F.col("year") == 2024)

N_TRAIN, N_VAL, N_TEST = train.count(), val.count(), test.count()
print(f"train {N_TRAIN:,}   val {N_VAL:,}   test {N_TEST:,}")
assert (N_TRAIN, N_VAL, N_TEST) == (34_453_808, 6_719_274, 6_374_476)

N_DRY = train.filter(F.col("label") == 0).count()    # dry rows before any sampling
N_WET = N_TRAIN - N_DRY
print(f"train dry {N_DRY:,}   train wet {N_WET:,}")
assert (N_DRY, N_WET) == (32_379_695, 2_074_113)

train 34,453,808   val 6,719,274   test 6,374,476
train dry 32,379,695   train wet 2,074,113


94 % of training rows have label 0 (no rain in the next 30 minutes), and most of them are quiet, dry hours that look alike. With `SUBSAMPLE = True`, `sampleBy` keeps every wet row and each dry row with probability `FRAC`. 

Without correction the model would see 39 % wet rows instead of 6 % (for `FRAC = 0.1`) and would over-predict rain. Each kept dry row stands for `1 / FRAC` original rows, so it gets weight `w = 1 / FRAC` (inverse-probability weighting). 

In [33]:
if SUBSAMPLE:
    fractions = {float(k): (FRAC if k == 0 else 1.0) for k in range(9)}
    train = train.sampleBy("label", fractions=fractions, seed=SEED)
    train = train.withColumn("w", F.when(F.col("label") == 0, 1.0 / FRAC).otherwise(1.0))
else:
    train = train.withColumn("w", F.lit(1.0))

val  = val.withColumn("w", F.lit(1.0))
test = test.withColumn("w", F.lit(1.0))

In [34]:
def assemble(df):
    out = (df.join(F.broadcast(static), on="station_id", how="inner")
             .transform(add_time_features))
    feature_cols = [c for c in out.columns if c.startswith("t0_")]
    return out.select("station_id", "slot", "ts", "year",
                      "y_mm", "label", "w", "past30_mm", "regime", *feature_cols)

FEATURES_T0 = [c for c in assemble(val.limit(1)).columns if c.startswith("t0_")]
print(len(FEATURES_T0), FEATURES_T0)
# Below are all T0 features
assert len(FEATURES_T0) == 15

15 ['t0_x_km', 't0_y_km', 't0_wet_ratio', 't0_heavy_ratio', 't0_hod_s1', 't0_hod_c1', 't0_hod_s2', 't0_hod_c2', 't0_hod_s3', 't0_hod_c3', 't0_doy_s1', 't0_doy_c1', 't0_doy_s2', 't0_doy_c2', 't0_season']


In [35]:
# Write the three tables
OUTPUTS = {"train": ASSEMBLED / TRAIN_NAME, "val": ASSEMBLED / "val", "test": ASSEMBLED / "test"}
for name, df in [("train", train), ("val", val), ("test", test)]:
    assemble(df).write.mode("overwrite").parquet(str(OUTPUTS[name]))
    print("written", name, "->", OUTPUTS[name].name)

written train -> train_frac0.1


written val -> val


written test -> test


In [ ]:
tables = {name: spark.read.parquet(str(path)) for name, path in OUTPUTS.items()}
is_wet = (F.col("label") > 0).cast("double")

rows = []
for name, t in tables.items():
    r = t.agg(F.count("*").alias("rows"),
              F.sum(is_wet).alias("wet_rows"),
              F.sum("w").alias("sum_w"),
              (F.sum(F.col("w") * is_wet) / F.sum("w")).alias("weighted_wet_share")).first()
    rows.append({"table": name, **r.asDict()})
summary = pd.DataFrame(rows).set_index("table")
summary

,rows,wet_rows,sum_w,weighted_wet_share
table,,,,
train,5314338,2074113.0,34476363.0,0.060160
val,6719274,449152.0,6719274.0,0.066845
test,6374476,405430.0,6374476.0,0.063602


: 

Note: in the following task, SHOULD read `val/`, `test/` and one training folder. Set the training folder there to the `TRAIN_NAME` printed at the top of this notebook (`train_frac0.1` or `train_full`), and always pass `weightCol="w"` to the models.